In [1]:
from pathlib import Path
import pandas as pd


def build_well_prefix_from_filename(filename: str) -> str:
    """
    From a filename like:
        LRS70_D_YSI_R_20250226_processed.csv
    build:
        LRS70D

    It takes the first two underscore-separated parts and concatenates them.
    """
    stem = Path(filename).stem
    parts = stem.split("_")

    if len(parts) < 2:
        raise ValueError(
            f"Filename does not have at least two underscore-separated parts: {filename}"
        )

    return f"{parts[0]}{parts[1]}"


def find_best_vadose_match(well_prefix: str, vadose_df: pd.DataFrame) -> pd.Series | None:
    """
    Find the best matching row in vadose_df for a given well_prefix.

    Match rule:
    - A row matches if the 'Well' value starts with well_prefix.
      Example: LRS69DR matches LRS69D.

    Priority rules:
    1) If multiple matches exist, use the first one with a non-null vz_thickness_m_2025
    2) If none has vz_thickness_m_2025, use the first one with a non-null vz_thickness_m_2023
    3) If matches exist but both values are null in all of them, return the first match
    4) If no matches exist, return None
    """
    if "Well" not in vadose_df.columns:
        raise KeyError("Column 'Well' not found in vadose zone reference file.")

    matches = vadose_df[
        vadose_df["Well"].astype(str).str.strip().str.upper().str.startswith(well_prefix.upper(), na=False)
    ]

    if matches.empty:
        return None

    # First priority: first row with vz_thickness_m_2025
    if "vz_thickness_m_2025" in matches.columns:
        matches_2025 = matches[matches["vz_thickness_m_2025"].notna()]
        if not matches_2025.empty:
            return matches_2025.iloc[0]

    # Second priority: first row with vz_thickness_m_2023
    if "vz_thickness_m_2023" in matches.columns:
        matches_2023 = matches[matches["vz_thickness_m_2023"].notna()]
        if not matches_2023.empty:
            return matches_2023.iloc[0]

    # If there are matches but all candidate values are empty, return first match
    return matches.iloc[0]


def get_vz_value(match_row: pd.Series) -> float | None:
    """
    Return vadose zone thickness using:
    1) vz_thickness_m_2025
    2) fallback to vz_thickness_m_2023
    3) None if neither exists or both are null
    """
    vz_2025 = match_row.get("vz_thickness_m_2025", pd.NA)
    if pd.notna(vz_2025):
        return float(vz_2025)

    vz_2023 = match_row.get("vz_thickness_m_2023", pd.NA)
    if pd.notna(vz_2023):
        return float(vz_2023)

    return None


def correct_conductivity_profiles_with_vadose_zone(
    profiles_dir: str | Path,
    root_dir: str | Path,
    output_dir: str | Path | None = None,
    depth_column: str = "Vertical Position m",
) -> None:
    """
    Read conductivity profile CSVs from profiles_dir, match them to the vadose zone
    reference file at:
        root_dir / data / vadose_zone / data_source_reference.csv

    Then add the vadose zone thickness to the depth_column and save corrected CSVs
    with suffix '_vz'.

    Output:
    - If output_dir is None, saves next to the originals.
    - Otherwise saves into output_dir.

    Matching logic:
    - File: LRS70_D_YSI_R_20250226_processed.csv -> well_prefix = LRS70D
    - Search in 'Well' column of vadose table for strings starting with LRS70D
      (e.g. LRS70DR is also a match).

    Special cases:
    - If a match exists but vz_thickness_m_2025 is empty, use vz_thickness_m_2023
    - If multiple matches exist, use the first with vz_thickness_m_2025
    """
    profiles_dir = Path(profiles_dir)
    root_dir = Path(root_dir)

    vadose_file = root_dir / "data" / "vadose_zone" / "data_source_reference.csv"

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    if not profiles_dir.exists():
        raise FileNotFoundError(f"Profiles directory not found: {profiles_dir}")

    if not vadose_file.exists():
        raise FileNotFoundError(f"Vadose zone reference file not found: {vadose_file}")

    vadose_df = pd.read_csv(vadose_file)

    csv_files = sorted(profiles_dir.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in: {profiles_dir}")
        return

    print(f"Found {len(csv_files)} profile files.")
    print(f"Using vadose reference: {vadose_file}\n")

    for csv_file in csv_files:
        try:
            well_prefix = build_well_prefix_from_filename(csv_file.name)
        except Exception as e:
            print(f"[SKIP] {csv_file.name} -> could not build well prefix: {e}")
            continue

        match_row = find_best_vadose_match(well_prefix, vadose_df)

        if match_row is None:
            print(f"[SKIP] {csv_file.name} -> no match found for well prefix '{well_prefix}'")
            continue

        vz_value = get_vz_value(match_row)

        if vz_value is None:
            print(
                f"[SKIP] {csv_file.name} -> match found in Well='{match_row.get('Well', 'UNKNOWN')}', "
                "but both vz_thickness_m_2025 and vz_thickness_m_2023 are empty"
            )
            continue

        try:
            df = pd.read_csv(csv_file)
        except Exception as e:
            print(f"[SKIP] {csv_file.name} -> could not read CSV: {e}")
            continue

        if depth_column not in df.columns:
            print(f"[SKIP] {csv_file.name} -> column '{depth_column}' not found")
            continue

        df = df.copy()
        df[depth_column] = pd.to_numeric(df[depth_column], errors="coerce")
        df[depth_column] = df[depth_column] + vz_value

        out_name = f"{csv_file.stem}_vz.csv"
        out_path = (output_dir / out_name) if output_dir else (csv_file.parent / out_name)

        try:
            df.to_csv(out_path, index=False)
            print(
                f"[OK] {csv_file.name} -> {out_name} | "
                f"match='{match_row.get('Well', 'UNKNOWN')}' | vz={vz_value}"
            )
        except Exception as e:
            print(f"[SKIP] {csv_file.name} -> could not save output: {e}")



In [5]:
if __name__ == "__main__":
    # Example usage
    root_dir = Path(r"C:\Users\Mariana\Documents\freshwater_lens")  # repo root
    profiles_dir = root_dir / "data" / "rawdy" / "rawdy_sat51w2p_priority"


    correct_conductivity_profiles_with_vadose_zone(
        profiles_dir=profiles_dir,
        root_dir=root_dir,
        output_dir=root_dir / "data" /  "rawdy" / "rawdy_sat51w2p_priority_vz",
        depth_column="Vertical Position m",
    )

Found 5 profile files.
Using vadose reference: C:\Users\Mariana\Documents\freshwater_lens\data\vadose_zone\data_source_reference.csv

[OK] AW5_D_YSI_20250225_processed.csv -> AW5_D_YSI_20250225_processed_vz.csv | match='AW5D' | vz=0.571
[OK] AW6_D_YSI_20250226_processed.csv -> AW6_D_YSI_20250226_processed_vz.csv | match='AW6D' | vz=1.006
[OK] BW3_D_YSI_Y_20250222_processed.csv -> BW3_D_YSI_Y_20250222_processed_vz.csv | match='BW3D' | vz=3.321
[OK] LRS69_D_YSI_R_20250222_processed.csv -> LRS69_D_YSI_R_20250222_processed_vz.csv | match='LRS69DR' | vz=2.149
[OK] LRS70_D_YSI_R_20250226_processed.csv -> LRS70_D_YSI_R_20250226_processed_vz.csv | match='LRS70D' | vz=0.46
